# `ho_friction_inv`

> Generated from `config/ais_0.1.py` by `config/make_step_notebooks.py` on 2026-10-29. Edit the driver and regenerate; changes made here are overwritten.

HO friction re-inversion in chunks; each submission resumes from the last saved chunk.

**Reads** (`load_model` calls in this step):
- `resume_path = f'{model_dir}/AIS3_ho_friction_inv.nc'`
- `f'{model_dir}/AIS3_thermal_steadystate.nc'`

**Writes** (`save_model` calls in this step):
- `f'{model_dir}/AIS3_ho_friction_inv.nc'`

**Run as a batch job** (from `config/notebooks/`; resources copied from `config/launch_ho_friction_inv.pbs`):

```bash
qsub -q normal -l ncpus=8,mem=190GB,walltime=25:00:00 -v NB=ais_0.1/13_ho_friction_inv.ipynb launch_notebook.pbs
```

The executed copy is written next to this notebook as `executed/<name>.<jobid>.ipynb`. Interactive use needs a Jupyter session with at least those resources.

**Run options**: none.

Status, results and known failures: [`docs/run_guide.md`](../../../docs/run_guide.md) and the worklogs in `docs/`.

### What this step does
Inverts the HO basal friction (Schoof `C`) with m1qn3:
- cost functions 101/103 (velocity misfit) on **surface** vertices
- cost function 501 (smoothness of C) on **base** vertices

Each submission runs `maxsteps=5` iterations and saves `AIS3_ho_friction_inv.nc`. The next
submission resumes from that file, copying `results.StressbalanceSolution.FrictionC` into
`md.friction.C`. Without a checkpoint it starts from `AIS3_thermal_steadystate.nc`.

### Status
- Paused after chunk 3. Chunk 4 failed five times (hangs at iteration 2, one SIGBUS;
  worklog §3.4, §4h, §4j).
- Grounded RMSE is 94–99 m/yr.
- The file on disk is chunk 3 with targeted smoothing. The unsmoothed chunk 3 is
  `AIS3_ho_friction_inv_chunk3_backup_presmoothing.nc`.
- This field was inverted with HO's old, too-soft B, from before the D2 rescale (worklog
  §5c/§5d).

### Before you run
- Copy the current checkpoint to a chunk-tagged name first. A completed chunk overwrites it,
  and a walltime-killed chunk saves nothing.
- The inner solve runs 96 hugemem cpus, which is capped at 24 h.

### What to check afterwards
- The m1qn3 cost table in `execution/AIS3_ho_friction_inv/*.outlog`:
  - Iteration 1 should continue the previous chunk's final cost. If it shows `3.0065e7`
    again, the chunk restarted from scratch (the resume bug).
  - The cf101/cf103 contributions should not be 0/0. If they are, the misfit mask is wrong.
- The printed grounded (surface) RMSE.

### Known pitfalls
- After a restart, iteration 2 can jump 12–19× in cost (m1qn3 loses its curvature estimate
  between submissions). `dfmin_frac=0.3` targets this.
- Tightening `restol` to 0.001 made single iterations run past 21 h.

_Hand-written notes: `config/notebooks/notes/ho_friction_inv.md` (edit there, then regenerate)._

## Setup

The preamble of `ais_0.1.py`: imports, helpers, settings and the data catalog.

#### Imports

In [ ]:
import pyissm
import ccdtools as ccdtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
import xarray as xr
import os
import glob
import copy
from pathlib import Path

#### `racmo_mean_smb()` — Time-mean RACMO2.4p1 smbgl over calendar years y0..y1 on md's mesh, in m ice eq/yr

In [ ]:
def racmo_mean_smb(md, y0, y1):
    """Time-mean RACMO2.4p1 smbgl over calendar years y0..y1 on md's mesh, in m ice eq/yr.

    Same unit handling as historical_dhdt_tuning (monthly kg/m^2 summed, divided by rho_ice),
    averaged over the years. Uses the module-level `catalog`.
    """
    data = catalog.load_dataset('racmo2.4p1_monthly_11km_1979-2023')
    smbgl = data['smbgl']
    rx, ry = pyissm.tools.general.ll_to_xy(data['lat'].values, data['lon'].values, -1)
    sel = smbgl.sel(time=(smbgl['time.year'] >= y0) & (smbgl['time.year'] <= y1))
    n_years = y1 - y0 + 1
    assert sel.sizes['time'] == 12 * n_years, f"expected {12 * n_years} months, got {sel.sizes['time']}"
    mean_myr = (sel.sum('time') / n_years / md.materials.rho_ice).squeeze('height').to_numpy()
    out = pyissm.data.interp.points_to_mesh(rx, ry, mean_myr, md.mesh.x, md.mesh.y)
    n_nan = int(np.isnan(out).sum())
    print(f"   RACMO {y0}-{y1} mean SMB: mesh-mean {np.nanmean(out):.3f} m ice eq/yr, "
          f"{n_nan} vertices outside RACMO coverage set to 0")
    return np.nan_to_num(out, nan=0.0)

#### `sync_mesh_z()` — Recompute the extruded mesh's vertex z from the current geometry (base + sigma*H)

In [ ]:
def sync_mesh_z(md):
    """Recompute the extruded mesh's vertex z from the current geometry (base + sigma*H).

    Keeps each column's relative layer spacing (sigma from the existing z), so it works even if
    md.mesh.z is stale -- e.g. after geometry was edited without touching z, as the thickness
    restoration (restore_floored_thickness.py) and the historical chunk resume did.
    """
    nv = md.mesh.numberofvertices
    nvb = int(md.mesh.numberofvertices2d)
    n_layers = nv // nvb
    assert n_layers * nvb == nv
    assert np.asarray(md.mesh.vertexonbase).astype(bool).ravel()[:nvb].all()
    assert np.asarray(md.mesh.vertexonsurface).astype(bool).ravel()[-nvb:].all()
    z = np.asarray(md.mesh.z, dtype=float).ravel().reshape(n_layers, nvb)
    span = z[-1] - z[0]
    safe = np.abs(span) > 1e-9
    sigma = np.where(safe[None, :], (z - z[0]) / np.where(safe, span, 1.0)[None, :],
                     np.linspace(0.0, 1.0, n_layers)[:, None])
    base = np.asarray(md.geometry.base, dtype=float).ravel()[:nvb]
    thick = np.asarray(md.geometry.thickness, dtype=float).ravel()[:nvb]
    d_base = np.abs(z[0] - base)
    d_surf = np.abs(z[-1] - (base + thick))
    print(f"   mesh.z vs geometry before sync: base max {d_base.max():.3f} m ({int((d_base > 0.01).sum())} "
          f"columns off by >1 cm), surface max {d_surf.max():.3f} m ({int((d_surf > 0.01).sum())} columns)")
    md.mesh.z = (base[None, :] + sigma * thick[None, :]).ravel()

#### `adopt_final_state()` — Make the transient's last step the model's own state before save_model

In [ ]:
def adopt_final_state(md):
    """Make the transient's last step the model's own state before save_model.

    solve() fills md.results only, so without this save_model writes the transient's INPUT
    geometry -- the reason neither relaxation ever reached downstream steps (worklog 4y).
    Carries Thickness and MaskOceanLevelset; base/surface are recomputed from them with the
    hydrostatic-floating/bed-grounded formula (raw output Base can sit below bed at newly
    grounded vertices, worklog 4o); velocities go to md.initialization when saved.
    """
    ts = md.results.TransientSolution
    nv = md.mesh.numberofvertices

    def last(field):
        if isinstance(ts, list):
            return np.asarray(getattr(ts[-1], field), dtype=float).ravel().copy()
        raw = np.asarray(getattr(ts, field), dtype=float).ravel()
        return raw.reshape(raw.size // nv, nv)[-1].copy()

    thick = last('Thickness')
    ol = last('MaskOceanLevelset')
    bed = np.asarray(md.geometry.bed, dtype=float).ravel()
    ri = float(md.materials.rho_ice)
    rw = float(md.materials.rho_water)
    base = np.maximum(np.where(ol >= 0, bed, -thick * ri / rw), bed)
    old_thick = np.asarray(md.geometry.thickness, dtype=float).ravel()
    old_ol = np.asarray(md.mask.ocean_levelset, dtype=float).ravel()
    md.geometry.thickness = thick
    md.geometry.base = base
    md.geometry.surface = base + thick
    md.mask.ocean_levelset = ol
    set_vel = []
    for field in ('Vx', 'Vy', 'Vz', 'Vel'):
        try:
            setattr(md.initialization, field.lower(), last(field))
            set_vel.append(field)
        except (AttributeError, ValueError):
            pass
    d = thick - old_thick
    print(f"   adopted final transient state: mean|dH| {np.abs(d).mean():.3f} m, max|dH| "
          f"{np.abs(d).max():.2f} m; grounded/floating changed at "
          f"{100 * np.mean((ol >= 0) != (old_ol >= 0)):.3f}% of vertices; initialization set: {set_vel}")

#### `rescale_rheology_B_d2()` — Rescale md's 3D rheology_B column by column to a 2D reference B, keeping the vertical

In [ ]:
def rescale_rheology_B_d2(md, B_ref_2d):
    """Rescale md's 3D rheology_B column by column to a 2D reference B, keeping the vertical
    profile ("D2", worklog 5c).

    Grounded columns match the deformation-weighted column B, B_eff = (sum w B^-n / sum w)^(-1/n)
    with w = (1 - sigma)^(n+1) (sigma 0 at the bed), so the soft basal layers count as they do for
    shear flow; floating columns match the plain column mean (plug flow). HO's steady-state
    temperature makes its basal ice much softer than the reference, which made HO flow ~9% too
    fast; with this rescale a diagnostic solve gave grounded surface speed / observed median 1.00.
    Idempotent: a column already matching the reference gets scale 1.
    """
    nv = md.mesh.numberofvertices
    nvb = int(md.mesh.numberofvertices2d)
    nl = nv // nvb
    B_ref_2d = np.asarray(B_ref_2d, dtype=float).ravel()
    assert B_ref_2d.size == nvb, f'reference B has {B_ref_2d.size} values, base layer has {nvb}'
    z = np.asarray(md.mesh.z, dtype=float).ravel().reshape(nl, nvb)
    span = z[-1] - z[0]
    sig = (z - z[0]) / np.where(np.abs(span) > 1e-9, span, 1.0)[None, :]
    w = np.diff(sig, axis=0)
    wsum = w.sum(axis=0)
    Bh = np.asarray(md.materials.rheology_B, dtype=float).ravel().reshape(nl, nvb)
    n = float(np.unique(np.asarray(md.materials.rheology_n).ravel())[0])
    B_mean = (0.5 * (Bh[1:] + Bh[:-1]) * w).sum(axis=0) / np.maximum(wsum, 1e-12)
    wdef = (1.0 - sig) ** (n + 1.0)
    wd = 0.5 * (wdef[1:] + wdef[:-1]) * w
    A_mid = 0.5 * (Bh[1:] ** -n + Bh[:-1] ** -n)
    B_eff = ((wd * A_mid).sum(axis=0) / np.maximum(wd.sum(axis=0), 1e-30)) ** (-1.0 / n)
    grounded = np.asarray(md.mask.ocean_levelset).ravel()[:nvb] > 0
    scale = np.where(grounded, B_ref_2d / B_eff, B_ref_2d / B_mean)
    scale = np.where(wsum > 0, scale, 1.0)
    md.materials.rheology_B = (Bh * scale[None, :]).ravel()
    print(f"   D2 rheology rescale: grounded median {np.median(scale[grounded]):.3f} "
          f"(p10 {np.percentile(scale[grounded], 10):.3f}, p90 {np.percentile(scale[grounded], 90):.3f}), "
          f"floating median {np.median(scale[~grounded]):.3f}")

#### `friction_law_info()` — Return (control_parameter, field_attr, min_bound, max_bound) for md's friction law

In [ ]:
def friction_law_info(md):
    """Return (control_parameter, field_attr, min_bound, max_bound) for md's friction law.

    Schoof (regularized Coulomb) inverts 'FrictionC' (field md.friction.C); Budd/Weertman (the
    'default' class) inverts 'FrictionCoefficient' (field md.friction.coefficient). The saved
    friction class (set by friction_law in ais_0.1_param.py) is the single source of truth.
    """
    if type(md.friction).__name__ == 'default':   # Budd / Weertman power law
        # VALIDATED bounds [0.05, 900] for p=q=1 (grounded RMSE 61.4 unregularised / 60.4 with
        # cf501=0.0001, see ais_0.1_param.py). The earlier [0.1, 10] bounds were tuned for the
        # superseded p=q=3 law (u ~ C^-6, `friconly_nfix`, RMSE 98.9) -- under p=q=1 (u ~ C^-2)
        # fast ice needs a much larger C for the same resisting stress, and that ceiling pinned
        # 41% of the domain at C=10 the first time p=1 was tried with it.
        return 'FrictionCoefficient', 'coefficient', 0.05, 900
    # Schoof (regularized Coulomb): tested directly for the Siple Coast trunk deficit and ruled
    # out on physics grounds, not just numerics -- see docs/inversion_worklog.md section 5.4.
    # Coupled-domain adjoint inversion of FrictionC is unstable near the Coulomb cap regardless
    # of solver settings, and a forward-only sweep across the full documented Cmax range
    # (0.17-0.84) left the Siple Coast trunk ratio completely unchanged from Budd's. Kept here
    # only so friction_law='schoof' remains loadable; not the recommended path.
    return 'FrictionC', 'C', 0.05, 250 ** 2        # Schoof (regularized Coulomb)

#### `extract_friction_inversion_domain()` — Extract the friction-inversion subdomain with a floating/grounded ice-front boundary condition

In [ ]:
def extract_friction_inversion_domain(md):
    """Extract the friction-inversion subdomain with a floating/grounded ice-front boundary condition.

    Extracts ALL ice (ice_levelset_elements < 1, includes the ice-front elements), so the new
    mesh boundary coincides exactly with the true, contiguous ice margin -- not an arbitrary
    internal cut. extract() imposes Dirichlet (observed velocity) on every new boundary node by
    default (see Model.py: "Boundary conditions: Dirichlets on new boundary"); this is reverted
    to Neumann (NaN spc) at boundary nodes classified as floating (ocean_levelset < 0), i.e. true
    ice-shelf calving fronts, where the natural ocean-pressure BC is physically correct. Boundary
    nodes classified as grounded (ocean_levelset >= 0) -- both marine-terminating (bed below sea
    level, no shelf) and true land-terminating (bed above sea level, no ocean to push back
    against) -- keep extract()'s default Dirichlet, since Neumann has no obvious physical meaning
    there. Classification is per-vertex (mds.mask.ocean_levelset), not per-element, so it follows
    the true ice-front geometry exactly with no fragmentation.

    A prior version anchored only the Ronne-Filchner/Ross fronts (the two largest floating
    regions, found to blow up under pure Neumann at low friction coefficient) and left everything
    else -- including land-terminating margins -- as Neumann. That fixed Ronne-Filchner/Ross but
    left land-terminating margins with a physically meaningless Neumann BC, which was the actual
    cause of a ~1e10 m/yr blowup at coeff=1 (confirmed: switching those margins to Dirichlet here
    brought coeff=1 down to ~1.8e7 m/yr).
    """
    ice_levelset_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ice_levelset)
    mds = md.extract(ice_levelset_elements < 1)

    bnd = mds.mesh.vertexonboundary.astype(bool)
    ocean_ls = np.asarray(mds.mask.ocean_levelset).ravel()
    floating_bnd = bnd & (ocean_ls < 0)

    mds.stressbalance.spcvx[floating_bnd] = np.nan
    mds.stressbalance.spcvy[floating_bnd] = np.nan
    mds.stressbalance.spcvz[floating_bnd] = np.nan
    mds.mask.ice_levelset[floating_bnd] = 0

    return mds

#### `load_shelf_rheology_B()` — Load the validated floating-shelf rheology inversion result (extractedvertices, B)

In [ ]:
def load_shelf_rheology_B():
    """Load the validated floating-shelf rheology inversion result (extractedvertices, B).

    execution_newB_rheology/run_001_1_10_1e-17 (2026-07-20) supersedes
    models/AIS3_ssa_rheology_floating_inv_lcurve/run_004_1_10_1e-17 (2026-06-30, the
    `rheology_lcurve_run` config value): same regularisation point (cf101=1, cf103=10,
    cf502=1e-17), recomputed later in this project after several geometry/N-flooring fixes
    were developed (100m thickness floor, N re-flooring against it, etc.) -- run_004 predates
    those fixes. This is the actual source the validated grounded-RMSE-60.4 friction result
    was warm-started from; loading the stale run_004 instead was found (via a direct A/B
    test) to reproduce RMSE ~114, not ~60 -- see docs/inversion_worklog.md. Not yet promoted
    into the canonical models/AIS3_ssa_rheology_floating_inv_lcurve/ directory, so this loads
    it from its original ad-hoc execution directory via solve(load_only=True) instead of
    io.load_model().
    """
    _cl = pyissm.model.classes.cluster.gadi()
    _cl.codepath = os.environ['ISSM_DIR'] + '/bin'
    _cl.executionpath = '/g/data/au88/jh7060/ACCESS-AIS3/execution_newB_rheology'
    _cl.login = 'jh7060'; _cl.project = 'au88'; _cl.storage = 'gdata/au88'

    mshelf = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')
    mshelf.mask.ice_levelset = pyissm.model.param.kill_icebergs(mshelf)
    sel = (mshelf.mask.ocean_levelset < 0) & (mshelf.mask.ice_levelset < 0)
    mshelf = mshelf.extract(sel)
    mshelf.cluster = _cl
    mshelf.settings.waitonlock = 0
    mshelf.inversion.iscontrol = 0
    mshelf.miscellaneous.name = 'run_001_1_10_1e-17'
    mr = pyissm.model.execute.solve(mshelf, 'Stressbalance', load_only = True, runtime_name = False, check_consistency = False)
    return np.asarray(mr.mesh.extractedvertices).ravel(), np.asarray(mr.results.StressbalanceSolution.MaterialsRheologyBbar).ravel()

## ------------------------------------
## Configure options
## ------------------------------------

#### Settings

In [ ]:
# Change directory to gdata to prevent storage limits in $HOME
os.chdir('/g/data/au88/jh7060/ACCESS-AIS3/')
os.environ['ISSM_DIR'] = '/g/data/vk83/apps/spack/1.1/release/linux-x86_64/issm-git.2026.05.18_2026.05.18-kgta35igm37z4qnqnul7rcmgx2inftqd'

# Should plots be generated?
plot = True
diagnostics = True
save = True
inversion_sensitivity = False

# Define execution directory
execution_dir = '/g/data/au88/jh7060/ACCESS-AIS3/execution'

# Define location to save final models
model_dir = '/g/data/au88/jh7060/ACCESS-AIS3/models'

# Define domain_file
domain_file = ('/g/data/au88/jh7060/ACCESS-AIS3/assets/ais_domain.exp')

# Define param_file
param_file = ('/g/data/au88/jh7060/ACCESS-AIS3/config/ais_0.1_param.py')

# Define cluster requirements
cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR']+'/bin'
cluster.executionpath = execution_dir
cluster.storage = 'gdata/au88+gdata/vk83'
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm_ad/2026.05.0']  # was access-issm/2025.11.0: executing a
# 2026.05.18 binary under a 2025.11.0 module load -- a stale-module mismatch caught and fixed
# across every scratchpad script this session; production had not been updated to match.
# np/memory: 32 cores / 100GB is under-provisioned -- this mesh needs ~130GB minimum even at
# 32 ranks (see docs/inversion_worklog.md section 8), which is the likely real cause of the
# OOM history noted below on the maxsteps line, not maxsteps itself. 48 cores / 190GB is the
# configuration validated as SU-optimal this session (>96 cores was actively worse).
cluster.np = 48
cluster.memory = 190
cluster.time = 60*48
cluster.login = 'jh7060'
cluster.project = 'au88'

# List all steps for clarity
all_steps = [
    'process_domain',
    'mesh',
    'param',
    'ssa_rheology_floating_inv_sensit',
    'ssa_rheology_floating_inv_lcurve',
    # 'ssa_rheology_floating_inv',
    'ssa_friction_forward_check',
    'ssa_friction_forward_check_budd',
    'ssa_friction_inv_sensit',
    'ssa_friction_inv_lcurve',
    'ssa_friction_inv_reg_lcurve',
    'ssa_inverted_solve',
    'ssa_relaxation',
    'ho_thermal_steadystate',
    'ho_friction_inv',
    'melt_gamma_tuning',
    'ho_relaxation',
    'historical_dhdt_tuning',
    'projection_ssp',
]

# Define steps to run
# steps = ['process_domain', 'mesh', 'param']
# steps = ['mesh', 'param']
# steps = ['ssa_rheology_floating_inv_sensit']
# steps = ['ssa_rheology_floating_inv_lcurve']
# steps = ['ssa_friction_forward_check']
# steps = ['ssa_friction_forward_check_budd']
# steps = ['param']
# steps = ['ssa_friction_inv_lcurve']
# steps = ['ssa_inverted_solve']
# steps = ['ssa_relaxation']
# steps = ['param']
# steps = ['param']
# steps = ['ssa_inverted_solve']
# steps = ['ssa_relaxation']
steps = ['ho_friction_inv']
# Per-job override via `qsub -v AIS3_STEPS=a,b` -- lets a launcher pick its step without editing
# the shared `steps` above and holding that edit through a multi-day hugemem queue wait.
if os.environ.get('AIS3_STEPS'):
    steps = os.environ['AIS3_STEPS'].split(',')
# Suffix for historical_dhdt_tuning's checkpoint file and execution name, so a rerun
# (e.g. AIS3_HIST_TAG=_chunk1rerun) cannot overwrite the finished AIS3_historical_1995_2019.nc.
HIST_TAG = os.environ.get('AIS3_HIST_TAG', '')
# steps = ['ssa_friction_inv_lcurve']
# steps = ['ssa_friction_inv_sensit']

#### Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)

In [ ]:
## ------------------------------------
## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)
## ------------------------------------
# Floating-ice rheology B field taken from the rheology L-curve (cf502 regularisation).
rheology_lcurve_run = 'run_004_1_10_1e-17'

# Preferred 101/103 cost-function coefficients for the friction inversion.
# The cf101=1000/cf103=0.1 choice below (run_021, vel_rmse=960.5) came from a sensit sweep run
# against the C_init=10 dead-zone bug (see ais_0.1_param.py): with u ~ C^-6 and the model stuck
# at zero velocity everywhere, that sweep's vel_rmse was never measuring model skill (it was
# ~equal to RMS(v_obs) itself, i.e. the null model). Every cell in that grid is void.
# VALIDATED instead (grounded RMSE 98.9, `friconly_nfix`): cf101=10, cf103=100 -- log-weighted,
# so the slow interior (which absolute weighting like 1000/0.1 effectively ignores) contributes
# to the fit. 10/100 was carried through every successful run this pipeline is based on.
friction_cf101 = 10
friction_cf103 = 100

# Mirrors friction_law in ais_0.1_param.py -- that flag only lives inside ais_0.1_param.py's
# own exec-scope (set on md.friction when 'param' in steps calls parameterize(), see below),
# so it isn't otherwise visible here at module load time where friction_lcurve_run (needed by
# ssa_inverted_solve) is defined. Keep this in sync with ais_0.1_param.py by hand.
friction_law = 'schoof'  # 'schoof' or 'budd' -- must match ais_0.1_param.py

# Effective-pressure source for the friction law. coupling=2 (ISSM internal "uniform sheet"
# hydrology, clamped >= 0) matched or beat coupling=3 (Ehrenfeucht dataset + manual N floor) in
# the earlier *uniform-coefficient forward-check* sweep -- but the full floating/grounded-BC
# inversion sensit sweep told a different story: coupling=2 has a specific, severe pathology at
# certain coefficient cells (cf101=cf103=10 and cf101=cf103=1000 both spiked to vel_rmse~13,100,
# ~13x every other cell) that the simpler forward-check never happened to probe. coupling=3 was
# clean and outlier-free across the entire 25-cell grid (vel_rmse 962-1385, no spikes). Reverted
# to coupling=3 on the strength of that full-grid evidence. AIS3_param.nc itself is also built
# with friction_coupling=3 (see ais_0.1_param.py), so this now matches the param-file default.
friction_coupling = 3

# m1qn3's relative gradient-norm stopping tolerance (default 1e-4). ROOT CAUSE of every earlier
# p=q=1 "convergence" that silently never fit anything: under p=1's much gentler cost-function
# landscape than p=3's, ||g(X)||/||g(X0)|| falls below 1e-4 by iteration ~16, while the cost is
# still falling fast (not flattening) and the fit is nowhere near done -- a false stop, not a
# real one. Tightened well below anything that can trigger this early, so every successful run
# in this pipeline instead stops on dxmin (step-size), the genuine convergence criterion.
friction_inv_gttol = 1e-8

# Dirichlet-pin two known-unstable regions (Institute/Moller Ice Stream band + an isolated
# cluster near x~350km,y~-1933km) to observed velocity during the friction inversion, instead
# of leaving them free -- mirrors Felicity's own constrain_Budd.exp/constrain_Schoof.exp
# pattern (runme.m: Inversion_Friction_Budd/Schoof). Built from diag_schoof_blowup_v2.py's
# worst-25 grounded vertices (all sat at the 100m thickness floor with driving stress
# exceeding Cmax*N under Schoof -- see docs/inversion_worklog.md). Root cause of that specific
# blowup turned out to be the forced-Newton solver setting (isnewton=2), not geometry, so this
# flag is OFF by default until an actual A/B test shows it changes anything for Budd -- see
# ssa_friction_inv_reg_lcurve below.
use_constrain_regions = False
constrain_exp_file = '/g/data/au88/jh7060/ACCESS-AIS3/assets/constrain_Budd.exp'

# Grounded-ice friction C field, in two stages (see ssa_friction_inv_lcurve /
# ssa_friction_inv_reg_lcurve below):
#   1. `friction_baseline_run` -- the UNREGULARISED (cf501 effectively off) p=q=1 baseline,
#      grounded RMSE 61.4, used only as the warm-start for stage 2 below (its own C field is
#      usable but ~5x rougher, C-field roughness 0.82 vs the p=q=3 baseline's 0.17).
#   2. `friction_lcurve_run` -- warm-started from (1), light DragCoefficientAbsGradient
#      regularisation (cf501). cf501=0.0001 is the validated corner: it drops the C-field
#      roughness to 0.18 (matching p=q=3) while the RMSE *improves* further, to 60.4 -- not a
#      tradeoff, both axes move the same direction. This is the field `ssa_inverted_solve` uses.
#
# The Budd naming pattern above (run_001_{cf101}_{cf103}_{cf501}) is specific to the Budd
# L-curve sweep; it does not apply to the Schoof m1qn3 continuation run (different control
# parameter, different script, not a cf501 grid point), so this is branched on friction_law
# rather than reused. See friction_law in ais_0.1_param.py for the full rationale for the
# current Schoof choice; ssa_friction_inv_reg_lcurve has never actually been run for Schoof --
# this points at the scratchpad-run tight-restol result saved into this same directory
# structure by finalize_schoof_friction_result.py, not a production-pipeline output.
if friction_law == 'schoof':
    friction_baseline_run = 'schoof_m1qn3_tightrestol_cmax2.0'
    friction_lcurve_run = 'schoof_m1qn3_tightrestol_cmax2.0'
else:
    friction_baseline_run = f'run_001_{friction_cf101}_{friction_cf103}_1e-08'
    friction_lcurve_run = f'run_001_{friction_cf101}_{friction_cf103}_0.0001'

#### Initialise Data Catalog

In [ ]:
## ------------------------------------
## Initialise Data Catalog
## ------------------------------------
catalog = ccdtools.catalog.DataCatalog()
bedmachine_data = catalog.load_dataset('measures_bedmachine_antarctica', version = 'v3')
velocity_data = catalog.load_dataset('measures_insar_based_antarctica_ice_velocity_map', version = 'v2')
measures_coastline = catalog.load_dataset('measures_antarctic_boundaries', subdataset = 'coastline')

## Step `ho_friction_inv`

In [ ]:
steps = ['ho_friction_inv']  # this notebook's step; overrides the driver default and AIS3_STEPS

**Section comment from the driver** (kept as written; it records history and may be out of date):

### Stage 2 (SCAFFOLD): Higher-order friction re-inversion

SSA-tuned friction (from ssa_friction_inv_reg_lcurve) isn't physically valid once HO adds vertical-shear resistance to the force balance -- the friction coefficient has to absorb a different share of the driving stress. Warm-start from the SSA-inverted C (same warm-start pattern validated for every mesh/geometry change this session) rather than cold-starting from C_init, and reuse the existing sensitivity-sweep infrastructure, which is confirmed to just call a generic Stressbalance solve (respects whatever flow equation is already set on md, not SSA-hardcoded) -- but this combination (HO + m1qn3 inversion via parameter_sensitivity) has not been tested. Verify on the same small test region as stage 1 before trusting continent-wide.

In [ ]:
print("-------------------------------------------------------------")
print(f" HIGHER-ORDER (HO) FRICTION RE-INVERSION"                  )
print("-------------------------------------------------------------")

# Chunked warm-restart: the first full-continental attempt (96 cores/24h) made real
# progress -- m1qn3 cost dropped 3.01e7 -> 3.21e6 (89%) over 12 iterations -- but was
# killed by NCI's walltime cap mid-iteration-13 with NO checkpoint saved (solve() only
# writes results after the mpiexec process returns normally; ISSM's m1qn3 loop has no
# built-in mid-run checkpoint). All 12 iterations of progress were lost. Fix: cap each
# job's iteration budget well within the 24h wall clock (~2h/iteration observed, so
# maxsteps=8 -> ~16h with ~8h margin) and have each run resume from its own prior
# partial output if one exists, so repeated 24h submissions accumulate progress instead
# of re-doing it. Cost: m1qn3's internal quasi-Newton curvature estimate is NOT
# preserved across restarts (ISSM has no mechanism to serialize it), so each chunk
# starts a fresh approximation -- a real but unavoidable overhead of this workaround.
resume_path = f'{model_dir}/AIS3_ho_friction_inv.nc'
resuming = os.path.exists(resume_path)
if resuming:
    print(f"-- Resuming from partial HO friction inversion ({resume_path})...")
    md = pyissm.model.io.load_model(resume_path)
else:
    print(f"-- Loading thermal steady-state model (no partial inversion found -- fresh start)...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_thermal_steadystate.nc')

# Warm start is ALREADY carried through, not a separate step: AIS3_inverted.nc has the
# correctly SSA-solved FrictionCoefficient, ho_thermal_steadystate extrudes it, and
# Model.extrude() projects every existing 2D field (including friction, via
# md.friction._extrude(md), confirmed in pyISSM/model/Model.py) onto the new 3D mesh
# automatically -- so md.friction.coefficient here is already the warm-started field,
# just replicated up every vertical column rather than defined only at the base.
fric_control, fric_field, fric_min, fric_max = friction_law_info(md)

if resuming:
    # BUGFIX (found after chunk 2 silently re-ran chunk 1's exact trajectory --
    # iteration 1's cost matched chunk 1's iteration 1 to 4 significant figures,
    # 3.0065e7 both times): save_model()/load_model() round-trips md.friction.<field>
    # unchanged from whatever it was BEFORE the inversion -- the optimized value only
    # ever lives in md.results.StressbalanceSolution.<fric_control> (same convention
    # ssa_inverted_solve already handles explicitly when grafting a completed
    # inversion elsewhere in this file). Loading a checkpoint without this copy silently
    # restarts from the pre-chunk-1 field every time, burning a full chunk's compute
    # for zero progress.
    print(f"-- Copying the resumed chunk's optimized {fric_control} into the live friction field...")
    _resumed = np.asarray(getattr(md.results.StressbalanceSolution, fric_control)).ravel()
    setattr(md.friction, fric_field, _resumed)

#### Restricting friction control to base-layer vertices

In [ ]:
print(f"-- Restricting friction control to base-layer vertices...")
# Friction is a basal boundary condition -- only vertexonbase vertices are physically
# meaningful control points. Pin every non-base vertex's bounds to its current
# (replicated) value so m1qn3 doesn't spend gradient steps on physically meaningless
# upper-layer copies of the same nominal field. vertexonbase / numberofvertices2d are
# both set by Model.extrude() (pyISSM/model/Model.py:819,832).
nv3d = md.mesh.numberofvertices
on_base = np.asarray(md.mesh.vertexonbase).astype(bool).ravel()
print(f"   {int(on_base.sum())} of {nv3d} vertices are on the base layer "
      f"(expect == numberofvertices2d = {md.mesh.numberofvertices2d})")

_fld = getattr(md.friction, fric_field).astype(float)
md.inversion = pyissm.model.classes.inversion.m1qn3(md.inversion)
md.inversion.iscontrol = 1
md.inversion.control_parameters = [fric_control]
md.inversion.min_parameters = np.where(on_base, fric_min, _fld)
md.inversion.max_parameters = np.where(on_base, fric_max, _fld)
# Per-chunk budget, not the full convergence target -- see the resume/checkpoint note
# above. Reduced from 8 to 5 (2026-08-30, alongside the restol/maxiter fix just above):
# chunk 4 was killed at the 24h cap while stuck in a single non-converging nonlinear
# solve, and now that stressbalance.maxiter is raised to 300, an individual iteration
# that genuinely needs more nonlinear steps to converge will legitimately take longer
# than the old ~2h/iteration average -- 5 iterations leaves more margin against that
# than 8 did. gttol is still the real target: if a chunk reaches it before exhausting
# maxsteps, m1qn3 stops early and that's the converged answer; otherwise the next
# submission resumes and keeps going.
md.inversion.maxsteps = 5
md.inversion.maxiter = 5
md.inversion.gttol = friction_inv_gttol
# FIX (2026-09-02, after chunk 4 failed a THIRD time at the exact same point -- iteration
# 2's post-restart step, this time exceeding even a raised maxiter=300 without converging,
# ~2284 SU burned with zero checkpoint saved): dfmin_frac (pyissm's m1qn3 class,
# inversion.py:306, "expected reduction of cost function during the first step") was never
# set here, silently defaulting to 1.0 -- i.e. m1qn3 calibrates its very first step
# assuming it can cut cost by 100%, producing an oversized step. Every chunk restart loses
# m1qn3's curvature estimate (no serialization mechanism across resubmissions), so this
# exact miscalibration recurs at EVERY chunk boundary -- consistent with chunks 1 and 3
# both seeing large iteration-2 cost spikes (12-19x) that happened to recover, and chunk
# 4 (three separate attempts, different restol/maxiter each time) landing somewhere the
# forward nonlinear solve genuinely can't converge from. Both prior chunk-4 fixes
# (restol=0.001, then maxiter=300) only gave the solver more room to fail slowly and both
# failed -- this instead targets the actual mechanism: a much more conservative expected
# first-step reduction, so the post-restart step is properly scaled rather than
# overshooting.
md.inversion.dfmin_frac = 0.3

#### Assigning cluster and updating settings

In [ ]:
print(f"-- Assigning cluster and updating settings...")
md.miscellaneous.name = 'AIS3_ho_friction_inv'
# BUGFIX: same submission-never-happens pattern found and fixed in ssa_inverted_solve,
# ssa_relaxation, and ho_thermal_steadystate -- waitonlock=0 + load_only=True in the
# "if save" branch never submits (pyissm/model/execute.py:1078-1082 unconditional
# early return), it only loads results from an already-finished prior run. Fixed to
# the same synchronous submit-and-wait pattern used everywhere else in this pipeline.
# Also applying the hugemem fix proactively (found necessary for ho_thermal_steadystate's
# plain HO stress balance solve at this same 23.8M-node scale, which OOM-killed at
# 190GB/normal-queue) -- an m1qn3 inversion adds adjoint/gradient computation on top of
# the forward solve, so it will need at least as much memory, likely more.
md.cluster = pyissm.model.classes.cluster.gadi()
md.cluster.codepath = cluster.codepath
md.cluster.executionpath = cluster.executionpath
md.cluster.storage = cluster.storage
md.cluster.moduleuse = cluster.moduleuse
md.cluster.moduleload = cluster.moduleload
md.cluster.login = cluster.login
md.cluster.project = cluster.project
# TRIED and REVERTED (2026-08-27): normal queue at 768 cores/3072GB (16 nodes) to
# dodge hugemem's queue congestion -- OOM'd within 2 minutes (SIGKILL, 2.8TB used of
# 3.0TB requested; job 177583055). Confirmed via direct qsub testing that this is a
# dead end on `normal`, not just under-provisioned: (a) memory need is NOT flat with
# rank count -- it rose from 1.39-1.53TB@96 ranks (hugemem) to >2.8TB@768 ranks
# (normal), consistent with docs/inversion_worklog.md §8's per-rank-overhead finding;
# (b) Gadi's PBS ties memory strictly to a fixed ncpus/48-derived node count on
# `normal` -- a direct qsub test requesting ncpus=768/mem=6144GB (i.e. "same core
# count, more memory-only nodes") was rejected outright ("384.0GB per node exceeds
# what normal can provide"), so you cannot decouple node count from core count here.
# Net: getting more memory on `normal` requires more ranks, but more ranks need even
# more total memory -- a losing spiral, since `normal`'s 192GB/node ceiling is ~7.5x
# below hugemem's ~1450GB/node. hugemem's small rank count (96) avoids the
# rank-overhead growth entirely and comfortably fits the real ~1.4-1.5TB need in its
# 2900GB budget -- back to hugemem; the queue-wait is the lesser problem.
md.cluster.queue = 'hugemem'
md.cluster.np = 96
md.cluster.memory = 1450 * 2
md.cluster.time = 60 * 24
md.settings.waitonlock = 1440  # minutes

#### Setting-up cost function coefficients/masks

In [ ]:
print(f"-- Setting-up cost function coefficients/masks...")
# Start from the validated SSA values (cf101=10/cf103=100, grounded+observed mask,
# grounding-line-excluded 501 mask) -- same physical misfit/regularisation this project
# has used throughout, just re-evaluated under HO's own velocity field. cf501 is kept
# at the SSA-chosen 0.0001 as a starting point; HO's different force balance could
# shift the misfit-vs-regularisation tradeoff, so treat this as a prior, not a final
# value -- worth a small re-sweep (mirroring ssa_friction_inv_reg_lcurve's methodology)
# once this converges once and a baseline RMSE is in hand.
# BUGFIX (found via the PIG/Thwaites small-region test): cost functions 101/103 are
# 'SurfaceAbsVelMisfit'/'SurfaceLogVelMisfit' (pyissm/model/inversions.py:13-14) --
# velocity misfit against satellite-observed SURFACE speed, evaluated at surface
# vertices in a 3D model, not basal ones. This scaffold reused on_base (correct for
# the friction control's own bounds, since friction is a basal parameter) for the
# cost-function weighting mask too, which put nonzero coefficients at vertices where
# the misfit is never evaluated -- confirmed by the test showing contributions of
# exactly 0 for both 101 and 103 in the printed cost table (m1qn3 was optimizing pure
# regularisation smoothness, never actually fitting observed velocity at all). The 501
# regularisation term (DragCoefficientAbsGradient) genuinely is a basal-quantity
# gradient, so its mask correctly stays on_base.
on_surface = np.asarray(md.mesh.vertexonsurface).astype(bool).ravel()
vo = np.asarray(md.inversion.vel_obs).ravel()
ol = np.asarray(md.mask.ocean_levelset).ravel()
il = np.asarray(md.mask.ice_levelset).ravel()
mask = (vo > 0) & (ol >= 0) & on_surface
grounded_mask = (ol >= 0) & on_base
_elx = np.asarray(md.mesh.elements).astype(int) - 1
_float_v = ol < 0
_touch_f = _float_v[_elx].any(axis = 1)
_gladj = np.zeros(nv3d, dtype = bool)
_gladj[_elx[_touch_f].ravel()] = True
reg_mask = grounded_mask & ~_gladj

cf = np.zeros((nv3d, 3))
cf[mask, 0] = friction_cf101
cf[mask, 1] = friction_cf103
cf[reg_mask, 2] = 0.0001
md.inversion.cost_functions = [101, 103, 501]
md.inversion.cost_functions_coefficients = cf

md.transient = pyissm.model.classes.transient.deactivate_all(md.transient)
# BUGFIX (2026-08-30, after chunk 4 burned its full 24h budget stuck in the nonlinear
# solve, hitting "maximum number of nonlinear iterations (100) exceeded" 3x without
# recovering, zero checkpoint saved). This step never set stressbalance.maxiter, so it
# silently used ISSM's implicit default of 100 -- exactly the number in the "(100)
# exceeded" messages. Explicitly raised to 300 so a genuinely-converging-but-slow
# nonlinear solve has room to actually finish rather than being truncated mid-Newton.
#
# REVERTED 2026-08-30, then RE-REVERTED 2026-08-31: also tried tightening restol
# 0.01->0.001 at the same time, following the exact fix this project's own SSA-era
# p=q=1 friction inversion work used for a superficially similar symptom
# (docs/inversion_worklog.md, ais_0.1_ssa_friction_inv notebook: restol=0.01 was "too
# loose", letting the nonlinear solve settle into different Newton states for identical
# C and corrupting m1qn3's cost/gradient signal). That precedent was validated on a
# ~1.6M-vertex 2D SSA mesh; this is a 23.8M-node 3D HO mesh where the linear solve is
# already ~92% of total runtime (confirmed in earlier chunks' own timing breakdowns) --
# a 10x tighter tolerance costs far more here per extra Newton iteration. Symptom
# confirmed the concern directly: with restol=0.001, the retry's iteration 2 ran 21+
# hours without finishing (vs. the pre-fix chunks' spikes recovering within a handful
# of iterations inside budget) and was killed with nothing saved -- worse than the
# original failure this was meant to fix. The original chunk-4 failure is equally
# explained by simply "needed more than 100 iterations" (fixed by maxiter alone) as by
# "was converging to a corrupted state" (the SSA-era mechanism) -- no direct evidence
# here it was the latter. Reverted restol to 0.01; keeping maxiter=300, which is the
# uncontroversial part of the original fix (pure headroom, no added per-iteration cost).
md.stressbalance.restol = 0.01
md.stressbalance.maxiter = 300
md.stressbalance.reltol = 0.1
md.stressbalance.abstol = np.nan
md.settings.solver_residue_threshold = 1e-3

if save:
    print(f"-- Submitting and waiting on HO friction inversion...")
    md = pyissm.model.execute.solve(md, 'Stressbalance', load_only = False, runtime_name = False)

    if diagnostics:
        vel = np.asarray(md.results.StressbalanceSolution.Vel).ravel()
        # Surface-vs-surface, matching the cost function's own evaluation location and
        # the validated PIG/Thwaites methodology (base-vs-surface differs slightly due
        # to real HO vertical shear, confirmed via verify_rmse_data.py this session).
        gr = (il < 0) & (ol > 0) & on_surface
        print(f"\nHO FRICTION INVERSION DIAGNOSTICS:")
        print(f"   Grounded (surface) RMSE: {np.sqrt(np.nanmean((vel[gr]-vo[gr])**2)):.2f} m/yr")

    print(f"\nSaving to {model_dir}/AIS3_ho_friction_inv.nc")
    pyissm.model.io.save_model(md, f'{model_dir}/AIS3_ho_friction_inv.nc')
else:
    print(f"-- Submitting HO friction inversion...")
    md = pyissm.model.execute.solve(md, 'Stressbalance', load_only = False, runtime_name = False)